# 01 - Base de conhecimento e casos de teste
Baixa as leis federais (planalto.gov.br), quebra em trechos por artigo (com id, fonte, URL e licença) e valida os 50 casos de teste contra a base.
**Requisitos 01 (casos com 'não consta') e 02 (base em trechos com fonte).**

In [1]:
# Setup (Colab: Ambiente de execução > Alterar tipo > GPU T4). Local: só rode a partir da raiz do repositório.
import os, sys, subprocess
REPO = "https://github.com/melo-zz/calcada-livre-genai.git"   # <- troque pelo link do repositório público da equipe
if os.path.exists("/content"):                                  # estamos no Colab
    if not os.path.exists("/content/repo"):
        subprocess.run(["git", "clone", REPO, "/content/repo"], check=True)
    os.chdir("/content/repo")
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("pasta de trabalho:", os.getcwd())

pasta de trabalho: /content/repo


In [2]:
%pip install -q -r requirements.txt

In [3]:
import random, numpy as np
SEED = 42
random.seed(SEED); np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass

In [4]:
from src.kb import construir_base
trechos, resumo = construir_base("base")

  (usando cópia salva: fontes_html/l10098.html)
l10098      38 trechos
  (usando cópia salva: fontes_html/l13146.html)
l13146     201 trechos
  (usando cópia salva: fontes_html/l10048.html)
l10048       9 trechos
  (usando cópia salva: fontes_html/d5296.html)
d5296      110 trechos
  (usando cópia salva: fontes_html/ctb.html)
ctb        555 trechos
  (usando cópia salva: fontes_html/l10741.html)
l10741     138 trechos
  (usando cópia salva: fontes_html/l12587.html)
l12587      56 trechos
  (usando cópia salva: fontes_html/l13709.html)
l13709     151 trechos
Total: 1258 trechos em base/trechos.jsonl


## Tabela da base (fonte, formato, licença, nº de trechos)

In [5]:
import pandas as pd
tabela = pd.read_csv("base/fontes.csv")
tabela.loc[len(tabela)] = ["TOTAL", "", "", "", tabela["n_trechos"].sum()]
display(tabela)
tabela.to_markdown("base/tabela_fontes.md", index=False)
assert (tabela["n_trechos"][:-1] > 0).all(), "Alguma fonte falhou ao baixar: veja a mensagem ATENÇÃO acima (o site pode ter mudado a URL)."

,fonte,formato,licenca,url,n_trechos
0,Lei 10.098/2000 - Acessibilidade,HTML (planalto.gov.br),"Domínio público (art. 8º, IV, Lei 9.610/1998: ...",https://www.planalto.gov.br/ccivil_03/leis/l10...,38
1,Lei 13.146/2015 - Estatuto da Pessoa com Defic...,HTML (planalto.gov.br),"Domínio público (art. 8º, IV, Lei 9.610/1998: ...",https://www.planalto.gov.br/ccivil_03/_ato2015...,201
2,Lei 10.048/2000 - Atendimento prioritário,HTML (planalto.gov.br),"Domínio público (art. 8º, IV, Lei 9.610/1998: ...",https://www.planalto.gov.br/ccivil_03/leis/l10...,9
3,Decreto 5.296/2004 - Regulamenta as Leis 10.04...,HTML (planalto.gov.br),"Domínio público (art. 8º, IV, Lei 9.610/1998: ...",https://www.planalto.gov.br/ccivil_03/_ato2004...,110
4,Código de Trânsito Brasileiro,HTML (planalto.gov.br),"Domínio público (art. 8º, IV, Lei 9.610/1998: ...",https://www.planalto.gov.br/ccivil_03/leis/l95...,555
5,Lei 10.741/2003 - Estatuto da Pessoa Idosa,HTML (planalto.gov.br),"Domínio público (art. 8º, IV, Lei 9.610/1998: ...",https://www.planalto.gov.br/ccivil_03/leis/200...,138
6,Lei 12.587/2012 - Política Nacional de Mobilid...,HTML (planalto.gov.br),"Domínio público (art. 8º, IV, Lei 9.610/1998: ...",https://www.planalto.gov.br/ccivil_03/_ato2011...,56
7,Lei 13.709/2018 - LGPD,HTML (planalto.gov.br),"Domínio público (art. 8º, IV, Lei 9.610/1998: ...",https://www.planalto.gov.br/ccivil_03/_ato2015...,151
8,TOTAL,,,,1258


## Amostra de trechos

In [6]:
import random
for t in random.Random(SEED).sample(trechos, 4):
    print(t["id"], "|", t["fonte"], "|", t["url"]); print(t["texto"][:300], "\n")

l13146_art117 | Lei 13.146/2015 - Estatuto da Pessoa com Deficiência | https://www.planalto.gov.br/ccivil_03/_ato2015-2018/2015/lei/l13146.htm
Lei 13.146/2015, Art. 117. O art. 1º da Lei nº 11.126, de 27 de junho de 2005 , passa a vigorar com a seguinte redação:
“Art. 1º É assegurado à pessoa com deficiência visual acompanhada de cão-guia o direito de ingressar e de permanecer com o animal em todos os meios de transporte e em estabelecime 

l13146_art4 | Lei 13.146/2015 - Estatuto da Pessoa com Deficiência | https://www.planalto.gov.br/ccivil_03/_ato2015-2018/2015/lei/l13146.htm
Lei 13.146/2015, Art. 4º Toda pessoa com deficiência tem direito à igualdade de oportunidades com as demais pessoas e não sofrerá nenhuma espécie de discriminação.
§ 1º Considera-se discriminação em razão da deficiência toda forma de distinção, restrição ou exclusão, por ação ou omissão, que tenha o 

ctb_art126 | Código de Trânsito Brasileiro | https://www.planalto.gov.br/ccivil_03/leis/l9503compilado.htm
CTB

## Casos de teste com gabarito
O gabarito é por evidência: cada caso aponta uma frase que precisa existir na base; os trechos que a contêm são os 'relevantes'. Casos sem evidência são listados abaixo (corrija a frase em `avaliacao/casos.py` e rode de novo).

In [7]:
from avaliacao.casos import preparar
casos, invalidos = preparar(trechos)
print(f"{len(casos)} casos válidos | {sum(c['consta'] for c in casos)} 'consta' | {sum(not c['consta'] for c in casos)} 'não consta'")
print(f"{len(invalidos)} casos SEM evidência na base (excluídos):")
for c in invalidos:
    print(" -", c["pergunta"], "| evidência procurada:", repr(c["evidencia"]))

51 casos válidos | 41 'consta' | 10 'não consta'
0 casos SEM evidência na base (excluídos):


Se sobrarem poucos casos 'consta' (menos de ~35), a base não está com o texto esperado: me mande a lista acima que eu ajusto as evidências.